## Dataset fields

All fields are sourced from `song_ml_view`, which joins `songs` and `song_features`.

| Field | Type | Description |
|---|---|---|
| `song_id` | UUID | Primary key |
| `genre` | text | Genre label from MSD Tagtraum annotations |
| `tempo_bpm` | float | Tempo in beats per minute, extracted from MIDI tempo marks (defaults to 120 if absent) |
| `time_signature` | text | Time signature ratio string, e.g. `"4/4"`, `"3/4"` |
| `key_tonic` | text | Root note of the detected key — one of 12 pitch classes (`C`, `C#`, `D`, …) |
| `key_mode` | text | `"major"` or `"minor"` — best binary classification target |
| `key_confidence` | float | Pearson correlation from music21's Krumhansl-Schmuckler key-finding algorithm — measures how well the piece's pitch-class distribution matches the detected key's expected profile. Range 0–1. Values below ~0.7 suggest ambiguous tonality, modal content, or atonal writing; values above ~0.9 indicate strong adherence to a single key. |
| `song_hotttnesss` | float | MSD song-level popularity score (0–1). Derived from web listening data and play counts at the time of MSD release (~2011). `NULL` where MSD has no data (stored as 0.0 in the source). Useful as a continuous regression target. |
| `artist_hotttnesss` | float | MSD artist-level popularity score (0–1). Similar provenance to `song_hotttnesss` but aggregated at the artist level — more stable across songs from the same artist. `NULL` where MSD has no data. |
| `unique_chords` | int | Count of distinct chord roots across all measures |
| `total_measures` | int | Total number of measures (bars) in the song |
| `avg_note_density` | float | Average number of notes per measure |
| `chord_entropy` | float | Shannon entropy of the inferred chord root distribution across measures. **Caveat for single-instrument MIDIs**: chord roots are inferred from the aggregate pitch collection of each measure (all notes, regardless of timing), not from truly simultaneous notes. For melodic lines this is better interpreted as *tonal center variety per measure* — how much the implied pitch center shifts across the piece — rather than intentional harmonic richness. High values mean many pitch areas are emphasized across measures; low values mean the piece gravitates toward one or two tonal centers throughout. |
| `pitch_range` | int | Max MIDI pitch minus min MIDI pitch. Measures vertical span: 12 = one octave, 24 = two octaves, 36 = three octaves. Captures breadth but is sensitive to outlier notes — a single very high or low note inflates the value. |
| `avg_pitch` | float | Mean MIDI pitch number across all note events (MIDI 60 = middle C). Values in the 60–72 range sit in the middle register; higher values indicate a brighter/higher-register piece. Useful as a coarse proxy for instrument register. |
| `pitch_variance` | float | Statistical variance of all MIDI pitch values. Complements `pitch_range`: where range only captures the extremes, variance reflects how broadly pitches are distributed throughout the piece. A melody that mostly sits in the middle register but occasionally hits outliers will show high range but low variance. |
| `avg_note_duration` | float | Mean note duration in quarter beats |
| `note_duration_variance` | float | Variance of note durations — higher values indicate more rhythmic complexity |
| `melodic_interval` | float | Mean absolute semitone distance between consecutive notes, ordered by bar then beat position. Values near 1–2 indicate stepwise/scalar motion; values above 5–7 suggest angular, leaping lines. Note: large jumps at phrase boundaries inflate this value. |
| `note_count` | int | Total number of notes in the song |

## Gauge of piece complexity
| Feature | Direction | Dimension |
|---|---|---|
| `key_confidence` | lower | tonal/harmonic |
| `unique_chords` | higher | harmonic |
| `chord_entropy` | higher | harmonic |
| `pitch_range` | higher | melodic |
| `pitch_variance` | higher | melodic |
| `melodic_interval` | higher | melodic |
| `avg_note_density` | higher | melodic |
| `note_duration_variance` | higher | rhythmic |

**Note:** standardization required

## Suggestions:

### Hypothesis testing
key_mode gives you a natural split. Good questions with this data:

- Do major-key songs have higher tempo than minor? (tempo_bpm by key_mode)
- Is chord entropy higher in minor? (chord_entropy by key_mode)
- Is melodic interval larger in major or minor? (t-test or Mann-Whitney)

### Linear regression
- Predict tempo_bpm from the musical features — it's a real-valued, roughly continuous target and has no obvious direct causal relationship with pitch/chord features, which makes it interesting to model.

### Logistic regression
- is_major is the cleanest binary target you have. The pitch and chord features should have reasonable predictive signal for mode.

### Clustering
- Use the numeric feature columns (drop song_id, genre, key_mode, key_tonic, time_signature). Normalize first — tempo_bpm, note_count, and total_measures are on very different scales from entropy and melodic_interval.

### Inferential
- key_mode t-tests
  - Do minor-key songs have higher pitch_variance?
  - Do major-key songs have higher tempo_bpm?
  - Do key modes differ in avg_note_duration?

- genre: ANOVA, Kruskal-Wallis

- continuous targets for regression:
  - song_hotttness
  - artist_hotttness

  - do more complex songs hve less artist_hotttness

In [ ]:
# # Encode key_mode as binary — clean target for logistic regression
# df["is_major"] = (df["key_mode"] == "major").astype(int)

# # One-hot encode key_tonic (12 pitch classes) and time_signature
# df = pd.get_dummies(df, columns=["key_tonic", "time_signature"], drop_first=True)